[![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-orange?logo=googlecolab)](https://colab.research.google.com/github/YOUR_USERNAME/satellite-change-detection/blob/main/notebooks/04_resnet_backbone.ipynb)

# 04 ResNet Backbone

Upgrade the Siamese encoder to a shared pretrained ResNet-18 and fine-tune it safely.

**Prerequisites:** Google Colab GPU runtime, LEVIR-CD dataset access, repo files available.

**Expected runtime:** 45-75 minutes on a T4 GPU


## ResNet-18 Backbone Upgrade

This notebook swaps the custom encoder for a shared pretrained ResNet-18 backbone while keeping the decoder logic and change head compatible with dense prediction.

In [ ]:
from pathlib import Path
from copy import deepcopy
from src.factory import load_config, build_model

step3_config = deepcopy(load_config(Path('configs/full.yaml')))
step3_config['data']['root'] = '/content/levir_data'
step3_config['model']['use_cbam'] = False
step3_config['paths']['results_dir'] = 'results/step3'
step3_config['paths']['best_checkpoint_name'] = 'siamese_resnet_best.pth'
step3_config['paths']['history_prefix'] = 'step3'
model = build_model(step3_config)
print(model)
# Expected output: SiameseResNetUNet with a shared ResNet-18 encoder, decoder, and change head.


## Fine-Tuning Strategy

This training run freezes the ResNet backbone for the first five epochs, then unfreezes it and continues with separate learning rates for backbone versus decoder and change head parameters.

In [ ]:
from src.train import train_from_config

step3_summary = train_from_config(step3_config, config_name='step3_resnet18_no_cbam')
print(step3_summary['parameter_count'])
print(step3_summary['test_metrics'])
# Expected output: stronger F1 than Step 2, ideally above 0.65 once the ResNet backbone is wired correctly.


## Notebook Summary

The shared ResNet-18 encoder adds stronger pretrained features and a staged fine-tuning schedule, which should improve recall and boundary quality over the custom CNN encoder.